# Carvana: what happened to the cars we checked?

Read this notebook in order: **cohort/repeat coverage -> latest native statuses -> first Sold, repeats and
reappearances -> vehicles needing a check -> one evidence timeline**.
Historical CARFAX/title studies, old batch accounting and detailed audits are optional
sections after this main workflow. All settings are in one cell below.
It follows two fixed groups of VINs. Some are historical controls; they help us
understand the website but cannot supply new sales for the inventory pilot.

**Run All reads saved files only.** A website Sold label is an observation, not a
verified delivery or a final sale after returns. Start with [Notebook 20](20_carvana_history_analysis.ipynb)
for inventory and asking-price changes. This notebook adds the vehicle-page evidence.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if (ROOT / 'vehicle/src').is_dir():
    ROOT = ROOT / 'vehicle'
elif ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))


## Settings: cutoff, cohort inputs and optional studies

The default cutoff is now. For a historical view, set `AS_OF_OVERRIDE` before
running this notebook. Use a timestamp with a timezone, such as
`2026-09-10T02:00:00Z`. All displayed timestamps are UTC; elapsed ages are hours.

| Time | Meaning |
| --- | --- |
| `selected_at` | When a VIN became part of a fixed cohort |
| `checked_at` | When someone actually inspected its page |
| `available_at` | When the saved observation became available to this project |

Only membership and observations known by the cutoff are used. The two cohort
files are read as saved; this notebook does not select new vehicles.
`SHOW_AUDIT_DETAILS` controls full audit displays. Optional study calculations
remain available offline at the end; their dated paths are explicit here.


In [ ]:
PILOT_AS_OF = globals().get('AS_OF_OVERRIDE', pd.Timestamp.now(tz='UTC').isoformat())
PILOT_RECHECK_HOURS = globals().get('PILOT_RECHECK_HOURS_OVERRIDE', 24)
PILOT_CONFIGS = [Path(path) for path in globals().get('PILOT_CONFIGS_OVERRIDE', [
    ROOT / 'config/carvana_sale_pilot.json', ROOT / 'config/carvana_sale_pilot_extension_20260909.json'])]
INVENTORY_CONFIG = Path(globals().get('INVENTORY_CONFIG_OVERRIDE', ROOT / 'config/carvana_daily_tracking.json'))
INSPECT_RETAILER = globals().get('INSPECT_RETAILER_OVERRIDE', 'carvana')
INSPECT_HISTORY_VIN = globals().get('INSPECT_HISTORY_VIN_OVERRIDE', '5YJ3E1EA2PF544050')
FOLLOWUP_BATCH_LIMIT = globals().get('FOLLOWUP_BATCH_LIMIT_OVERRIDE', 12)
if type(FOLLOWUP_BATCH_LIMIT) is not int or not 1 <= FOLLOWUP_BATCH_LIMIT <= 12:
    raise ValueError('Choose a review batch limit from 1 to 12.')
# Optional dated studies; these paths never select the current status history.
HISTORY_STUDY_DIR = Path(globals().get('HISTORY_STUDY_DIR_OVERRIDE',
    ROOT / 'data/experiments/vehicle_history_study/20260910T034129Z'))
FOLLOWUP_PASS_DIR = Path(globals().get('FOLLOWUP_PASS_DIR_OVERRIDE',
    ROOT / 'data/experiments/prospective_followup/20260910T040311Z'))
EXISTING_ABSENCE_DAYS = 3  # Existing candidate rule; optional sensitivity only.
ABSENCE_SENSITIVITY_DAYS = [EXISTING_ABSENCE_DAYS, 7]
SHOW_AUDIT_DETAILS = globals().get('SHOW_AUDIT_DETAILS_OVERRIDE', False)
print('Evidence cutoff (UTC):', pd.Timestamp(PILOT_AS_OF).tz_convert('UTC').isoformat())
print('Frozen cohort sources:', *PILOT_CONFIGS, sep='\n')
print('Usable-native observation reminder interval, hours:', PILOT_RECHECK_HOURS)


In [ ]:
# Read-only replay; validate identities before any combined totals.
from vehicle_tracker.sale_pilot import known_disjoint_cohorts, load_pilot, summarize_pilot

sale_signal_study = pilot_observations = pilot_summary = pilot_queue = pd.DataFrame()
cohort_catalog = pd.DataFrame()
known_cohorts = []
alternative_view = any(name in globals() for name in [
    'CYCLE_REPORTS_OVERRIDE', 'TRACKING_CONFIG_OVERRIDE', 'DAILY_DATABASE_OVERRIDE',
    'DATABASE_OVERRIDE', 'CHECKS_OVERRIDE', 'REVIEWS_OVERRIDE', 'SALES_REVIEWS_OVERRIDE',
    'SELECTED_IDENTITY_OVERRIDE', 'CHECK_DRAFT_OVERRIDE'])
if alternative_view:
    print('Separate real pilot skipped for an alternative/synthetic analysis.')
else:
    missing_configs = [path for path in PILOT_CONFIGS if not path.is_file()]
    for path in missing_configs:
        print('Optional frozen cohort configuration is absent:', path, '| Its membership/coverage is unavailable.')
    frozen_cohorts = [json.loads(path.read_text(encoding='utf-8')) for path in PILOT_CONFIGS if path.is_file()]
    known_cohorts = known_disjoint_cohorts(frozen_cohorts, as_of=PILOT_AS_OF)
    known_ids = {cohort['cohort_id'] for cohort in known_cohorts}
    cohort_catalog = pd.DataFrame([dict(cohort_id=c['cohort_id'], selected_at=c['selected_at'],
        known_at_cutoff=c['cohort_id'] in known_ids) for c in frozen_cohorts])
    display(cohort_catalog)
    summaries, records = [], []
    for cohort in known_cohorts:
        observed = load_pilot(ROOT, cohort, as_of=PILOT_AS_OF)
        summary = summarize_pilot(observed, cohort, as_of=PILOT_AS_OF, recheck_hours=PILOT_RECHECK_HOURS)
        summary['cohort_id'] = cohort['cohort_id']
        summary['selected_at'] = cohort['selected_at']
        summary['selection_pending_group'] = summary.get('pending_group', pd.Series(index=summary.index, dtype='object')).fillna('not stratified')
        summaries.append(summary)
        if not observed.empty:
            records.append(observed.assign(cohort_id=cohort['cohort_id']))
    if summaries:
        pilot_summary = pd.concat(summaries, ignore_index=True)
        pilot_queue = pilot_summary[['cohort_id', 'retailer', 'vin', 'listing_id', 'url', 'role',
            'selection_pending_group', 'selection_reason', 'selected_at']].copy()
    if records:
        sale_signal_study = pd.concat(records, ignore_index=True)
        # Latest available interpretation per physical listing visit; source versions remain above.
        pilot_observations = sale_signal_study.sort_values(['checked_at', 'available_at']).drop_duplicates(
            ['cohort_id', 'retailer', 'vin', 'listing_id', 'checked_at'], keep='last')
    if not known_cohorts:
        print('No selected cohort was known at this cutoff. No outcome is assumed.')


## 1. Cohort membership and repeat-check coverage

Start with the whole selected group, including VINs with no saved visit. The
original and extension cohorts remain separate. Their native pending=true/false
selection groups are diagnostics, not a representative sample of Carvana.

A visit can fail or leave purchase availability unclear. `usable_native_checks`
means the VIN matched and Carvana's native Available/Sold field was readable.
`usable_status_checks` also requires a resolved page interpretation. These overlap;
do not add them together. A pre-order page can be native Available but still
unresolved about purchase readiness.

`prospective_repeat_observed` counts initially non-Sold prospective VINs with a
later usable native check; historical controls do not count.
The complete calculations remain in `pilot_coverage`. Measured sales and outcome
rates stay missing: an unchecked VIN does not mean zero sales.


In [ ]:
pilot_coverage = combined_coverage = no_retained_check = pd.DataFrame()
if not pilot_summary.empty:
    coverage_rows = []
    groups = ['cohort_id', 'role', 'selection_pending_group']
    for key, part in pilot_summary.groupby(groups, dropna=False, sort=False):
        followed = int(part.prospective_repeat_observed.sum())
        coverage_rows.append(dict(zip(groups, key), selected_VINs=len(part),
            no_retained_check=int(part.checks.eq(0).sum()), one_check=int(part.checks.eq(1).sum()),
            repeated_checks=int(part.checks.gt(1).sum()), observed_VINs=int(part.checks.gt(0).sum()),
            physical_checks=int(part.checks.sum()), usable_status_checks=int(part.usable_status_checks.sum()),
            usable_native_checks=int(part.usable_native_checks.sum()), access_failures=int(part.access_failures.sum()),
            unresolved_checks=int(part.unresolved_checks.sum()),
            VINs_without_usable_status=int(part.usable_status_checks.eq(0).sum()),
            initially_already_Sold=int(part.first_encountered_sold.sum()),
            prospective_repeat_observed=followed,
            first_observed_Sold_transitions=int(part.newly_observed_sold.sum()) if followed else pd.NA,
            measured_sales=pd.NA, outcome_rate=pd.NA))
    pilot_coverage = pd.DataFrame(coverage_rows)
    print('Coverage by cohort, role and native pending selection group:')
    with pd.option_context('display.max_columns', None):
        display(pilot_coverage[groups + ['selected_VINs', 'no_retained_check', 'one_check',
            'repeated_checks', 'usable_native_checks', 'access_failures', 'unresolved_checks']])
    # Cohort identities were validated as disjoint and selected by the cutoff before concatenation.
    coverage_count_columns = ['selected_VINs', 'observed_VINs', 'no_retained_check', 'one_check',
        'repeated_checks', 'physical_checks', 'usable_status_checks', 'usable_native_checks',
        'access_failures', 'unresolved_checks', 'prospective_repeat_observed']
    combined_coverage = pilot_coverage[coverage_count_columns].sum().to_frame('known disjoint cohorts').T
    combined_coverage['measured_sales'] = pd.NA
    combined_coverage['outcome_rate'] = pd.NA
    print('Combined coverage only; read cohort-specific follow-up before discussing outcomes:')
    display(combined_coverage[['selected_VINs', 'observed_VINs', 'no_retained_check',
        'physical_checks', 'usable_native_checks', 'access_failures']])
    no_retained_check = pilot_queue.merge(pilot_summary[['cohort_id', 'retailer', 'vin', 'checks']],
        on=['cohort_id', 'retailer', 'vin'], validate='one_to_one').query('checks == 0')
    print('Unchecked VINs are kept in no_retained_check and the attention table below.')


## 2. Latest native statuses and unresolved cases

`checked_vehicles` shows the **latest physical visit**, even if it failed. Native
`saleStatus` and `purchaseType` come from the target vehicle's public page data;
`latest_status` is our interpretation of those fields and its badge/button.
`why_unresolved` explains what the evidence still cannot establish.

| Native evidence | Interpretation |
| --- | --- |
| Sold with matching identity and consistent page evidence | `sold_label`: site-reported Sold |
| Available / NotPurchasable | `unavailable`: the cause is unknown |
| Available / Purchasable | `available` or `pending` only with a specific button/badge |
| Available / Reservable | Pre-order; native non-Sold, purchase readiness `unknown` |
| Failed access, conflicting identity or unrecognized fields | Status unresolved |

The parser checks URL, listing ID and VIN. It does not classify a recommendation
car or equipment text containing "as originally sold". Saved rows are selected
public-page projections, not original HTML or transaction records.


Intentional interpretation change: a complete target `hero_badge` matching
`On Hold` + newline + `MM:SS` is now `pending` when native Available/Purchasable
and the other UI evidence agree. The original badge (including `00:00`) stays
unchanged. This indicates purchase activity, not expiry or a completed sale.
Conflicting native/UI evidence still stays unresolved.


In [ ]:
latest_vehicle_status = checked_vehicles = pd.DataFrame()
if not pilot_summary.empty:
    latest_vehicle_status = pilot_summary.copy()
    latest_vehicle_status['why_unresolved'] = pd.NA
    latest_vehicle_status.loc[latest_vehicle_status.latest_status.eq('unavailable'), 'why_unresolved'] = (
        'The page is unavailable; it does not explain why.')
    unknown = latest_vehicle_status.latest_status.eq('unknown')
    latest_vehicle_status.loc[unknown, 'why_unresolved'] = 'No decisive purchase button or status badge.'
    latest_vehicle_status.loc[unknown & latest_vehicle_status.latest_purchaseType.eq('Reservable'), 'why_unresolved'] = (
        'Pre-order: native non-Sold, but purchase readiness is unclear.')
    failed = latest_vehicle_status.checks.gt(0) & latest_vehicle_status.latest_parse_outcome.ne('matched')
    latest_vehicle_status.loc[failed, 'why_unresolved'] = (
        'Check unresolved: ' + latest_vehicle_status.loc[failed, 'latest_parse_outcome'].fillna('unknown'))
    latest_vehicle_status.loc[latest_vehicle_status.checks.eq(0), 'why_unresolved'] = 'No retained page check.'
    checked_vehicles = latest_vehicle_status.loc[latest_vehicle_status.checks.gt(0)].copy()
    print('LATEST VISIT: website evidence, not completed sales. Full rows: checked_vehicles.')
    with pd.option_context('display.max_rows', None, 'display.max_colwidth', 65):
        display(checked_vehicles[['vin', 'role', 'latest_listing_id', 'checked_at',
            'latest_saleStatus', 'latest_purchaseType', 'latest_hero_badge', 'latest_status', 'why_unresolved']])
else:
    print('No cohort membership is available at this cutoff.')


## 3. First observed Sold transitions, repeated checks and reappearances

`pilot_changes` compares the previous and latest physical visits. A failed or
conflicting check leaves change flags missing; it cannot turn an earlier value
into a new vehicle state. A changed interpretation can simply reflect a missing
button while the native fields stay unchanged. The before/after values remain
visible, so we can investigate rather than assume a cancellation or sale.

A **first observed Sold transition** additionally requires a prior matched native
non-Sold check and a first Sold check at/after selection. Historical controls and
vehicles already Sold on their first check cannot supply that transition.
`transition_interval_hours` is the time between the bounding observations, not a
delivery date. Later reappearance is also an observation, not proof of a return.


`repeated_sold_checks` contains later matched Sold observations after the first
Sold observation for that retailer/VIN. These are repeat website checks; they
never increment `newly_observed_sold` or measured sales.


In [ ]:
pilot_changes = newly_observed_sold = initially_sold = later_reappearances = repeated_sold_checks = pd.DataFrame()
if not pilot_summary.empty:
    pilot_changes = pilot_summary.loc[pilot_summary.checks.gt(1)].copy()
    matched = pilot_changes.previous_parse_outcome.eq('matched') & pilot_changes.latest_parse_outcome.eq('matched')
    comparison_fields = [('saleStatus', 'previous_saleStatus', 'latest_saleStatus'),
                         ('purchaseType', 'previous_purchaseType', 'latest_purchaseType'),
                         ('interpretation', 'previous_status', 'latest_status')]
    change_columns = []
    for name, previous, latest in comparison_fields:
        column = name + '_changed'
        change_columns.append(column)
        pilot_changes[column] = pd.Series(pd.NA, index=pilot_changes.index, dtype='boolean')
        comparable = matched & pilot_changes[previous].notna() & pilot_changes[latest].notna()
        pilot_changes.loc[comparable, column] = pilot_changes.loc[comparable, previous].ne(pilot_changes.loc[comparable, latest])
    pilot_changes['change_observed'] = pilot_changes[change_columns].eq(True).any(axis=1)
    print('PREVIOUS / LATEST PHYSICAL VISITS: missing change flags mean not comparable.')
    with pd.option_context('display.max_rows', None):
        display(pilot_changes[['vin', 'previous_checked_at', 'checked_at',
            'previous_saleStatus', 'latest_saleStatus', 'previous_status', 'latest_status', *change_columns]])
    initially_sold = pilot_summary.loc[pilot_summary.first_encountered_sold,
        ['cohort_id', 'vin', 'role', 'first_sold_at', 'first_sold_listing_id']]
    newly_observed_sold = pilot_summary.loc[pilot_summary.newly_observed_sold,
        ['cohort_id', 'vin', 'role', 'selection_pending_group', 'last_non_sold_at', 'last_non_sold_listing_id',
         'first_sold_at', 'first_sold_listing_id', 'transition_interval_hours', 'reappeared_at', 'reappeared_listing_id']]
    print('Initially already-Sold vehicles (not new transitions):')
    display(initially_sold)
    print('Prospective VINs with a usable repeat check:', int(pilot_summary.prospective_repeat_observed.sum()))
    print('First qualifying Sold transitions; inspect follow-up coverage before interpreting an empty table:')
    display(newly_observed_sold)
    print('Read the observed/unobserved denominators above. This is not a count of actual sales.')
if not pilot_observations.empty:
    native_history = pilot_observations.loc[pilot_observations.parse_outcome.eq('matched')
        & pilot_observations.saleStatus.isin(['Available', 'Sold'])].sort_values(['checked_at', 'available_at']).copy()
    first_sold_time = native_history.checked_at.where(native_history.saleStatus.eq('Sold')).groupby(
        [native_history.cohort_id, native_history.retailer, native_history.vin]).transform('min')
    repeated_sold_checks = native_history.loc[native_history.saleStatus.eq('Sold')
        & native_history.checked_at.gt(first_sold_time),
        ['cohort_id', 'retailer', 'vin', 'listing_id', 'checked_at', 'available_at', 'saleStatus', 'source']]
    print('Repeated Sold checks after the first Sold observation (not additional sales):')
    display(repeated_sold_checks)
    native_history['purchasable_or_reservable'] = (native_history.checked_at.gt(first_sold_time)
        & native_history.saleStatus.eq('Available')
        & native_history.purchaseType.isin(['Purchasable', 'Reservable']))
    previous_eligible = native_history.groupby(['cohort_id', 'retailer', 'vin']).purchasable_or_reservable.shift(fill_value=False)
    later_reappearances = native_history.loc[native_history.checked_at.gt(first_sold_time)
        & native_history.purchasable_or_reservable & ~previous_eligible,
        ['cohort_id', 'retailer', 'vin', 'listing_id', 'checked_at', 'available_at', 'saleStatus', 'purchaseType']]
    print('Later observed reappearances, preserving each listing ID:')
    display(later_reappearances)


## 4. Vehicles needing another check

This is a **review list**, not an automatic collection command. It includes cars
with no check, failed/conflicting checks, changed website evidence, unresolved
page interpretations, or old native evidence. If several reasons apply, the
first reason in the visible `attention_order` list is shown.

`hours_since_usable_native` starts at the last matched native Available/Sold
observation. A recent failed visit does not refresh it. The 24-hour reminder is
an operating choice, not a sales rule. The full age diagnostics remain in
`pilot_freshness`. Follow-up links use the latest verified listing identity known
at the cutoff. The original cohort URL stays alongside it; failed checks cannot
replace a verified identity. Conflicting identity evidence leaves the link unresolved.


In [ ]:
from vehicle_tracker.daily import tracking_settings, tracking_history
from vehicle_tracker.events import _aware, vin_events

inventory_cycles = inventory_rows = inventory_source_rows = pd.DataFrame()
inventory_page_comparison = inventory_comparison_diagnostics = pd.DataFrame()
inventory_config = INVENTORY_CONFIG
if not pilot_summary.empty:
    if inventory_config.is_file():
        inventory_settings = tracking_settings(inventory_config)
        inventory_cycles, inventory_rows = tracking_history(inventory_settings, as_of=PILOT_AS_OF)
    else:
        print('Optional inventory config absent; no inventory observation is assumed.')
    if not inventory_cycles.empty:
        inventory_cycles = inventory_cycles.loc[inventory_cycles.available_at.map(_aware).le(_aware(PILOT_AS_OF))].copy()
        inventory_source_rows = inventory_rows.merge(inventory_cycles[['cycle_id', 'available_at']],
            on='cycle_id', validate='many_to_one').rename(columns={'available_at': 'inventory_available_at'})
        inventory_source_rows['inventory_observed_at'] = inventory_source_rows.observed_at_utc.map(_aware)
        inventory_source_rows['inventory_available_at'] = inventory_source_rows.inventory_available_at.map(_aware)

from vehicle_tracker.vehicle_history import followup_identities, FOLLOWUP_COLUMNS
history_followups = pd.DataFrame(columns=FOLLOWUP_COLUMNS)
identity_diagnostics = pd.DataFrame()
try:
    history_followups = followup_identities(pilot_queue, pilot_observations,
        inventory_source_rows, as_of=PILOT_AS_OF)
except ValueError as error:
    identity_diagnostics = pd.DataFrame([{'problem': str(error)}])
    print('Follow-up links unresolved; inspect identity_diagnostics.')
    display(identity_diagnostics)


In [ ]:
pilot_freshness = pilot_attention = pd.DataFrame()
if not pilot_summary.empty:
    pilot_freshness = pilot_summary[['cohort_id', 'retailer', 'vin', 'latest_listing_id',
        'last_attempt_at', 'hours_since_attempt', 'last_usable_native_at',
        'last_usable_native_listing_id', 'hours_since_usable_native',
        'no_usable_native_observation', 'overdue']].copy()
    attention = latest_vehicle_status.merge(history_followups, on=['retailer', 'vin'], how='left', validate='one_to_one')
    attention['attention_reason'] = ''
    # Later assignments take precedence; the order below makes that priority explicit.
    attention.loc[attention.overdue, 'attention_reason'] = 'Native evidence is overdue'
    attention.loc[attention.latest_status.isin(['unknown', 'unavailable']), 'attention_reason'] = 'Review unresolved page evidence'
    if not pilot_changes.empty:
        changed = pilot_changes.loc[pilot_changes.change_observed, ['cohort_id', 'retailer', 'vin']].assign(website_changed=True)
        attention = attention.merge(changed, on=['cohort_id', 'retailer', 'vin'], how='left', validate='one_to_one')
        attention.loc[attention.website_changed.eq(True), 'attention_reason'] = 'Website evidence changed'
    attention.loc[attention.checks.gt(0) & attention.latest_parse_outcome.ne('matched'), 'attention_reason'] = 'Review failed or conflicting check'
    attention.loc[attention.checks.eq(0), 'attention_reason'] = 'No retained check yet'
    attention_order = ['No retained check yet', 'Review failed or conflicting check',
        'Website evidence changed', 'Review unresolved page evidence', 'Native evidence is overdue']
    attention['_order'] = attention.attention_reason.map({reason: i for i, reason in enumerate(attention_order)})
    pilot_attention = attention.loc[attention.attention_reason.ne('')].sort_values(
        ['_order', 'checked_at', 'vin'], na_position='first').drop(columns='_order')
    print('Review order:', ' -> '.join(attention_order))
    with pd.option_context('display.max_rows', None, 'display.max_colwidth', 65):
        display(pilot_attention[['vin', 'role', 'listing_id', 'latest_listing_id', 'attention_reason',
            'hours_since_usable_native', 'original_url', 'followup_url']])


### Next bounded page-check batch

All selected cohort vehicles remain in `followup_plan`. Select **first checks**,
then **due repeats ordered by oldest `next_due_at`**, across prospective and
initially-Sold/control groups alike. Retailer, VIN and follow-up listing ID break
ties deterministically. Cohort and pending-selection groups remain analysis labels.

`next_due_at = last_usable_native_at + PILOT_RECHECK_HOURS`; a repeat is due when
that time is at or before `PILOT_AS_OF`. Never-checked vehicles can receive a first
check once their cohort is known. Not-yet-due vehicles never fill unused slots.
The default single-pass limit is 12; Settings accepts only 1?12, matching the importer.

A failed/conflicting latest attempt is deferred for review, even when the earlier
native clock is due. Attempts without any usable native evidence have no due time
and require review to establish a baseline. Failed attempts never reset the usable
clock. Unresolved follow-up identities are also deferred; no URL is guessed.
`needs_review`, `eligible_for_batch`, and `deferred_reason` show these distinctions.
An otherwise eligible vehicle can remain deferred because the batch limit was reached.

`next_followup_batch` and `followup_remainder` are disjoint and together account
for every cohort vehicle. A shorter or empty batch is valid. Run All only reads
and displays the plan; it neither collects nor authorizes a retry. Stop on an
access challenge, rate limit or identity conflict and review before resuming visits.


In [ ]:
followup_plan = next_followup_batch = followup_remainder = followup_group_counts = pd.DataFrame()
if not pilot_summary.empty:
    followup_plan = pilot_summary.merge(history_followups, on=['retailer', 'vin'],
        how='left', validate='one_to_one')
    # Preserve analytical groups; they do not set the repeat-check ordering.
    followup_plan['followup_group'] = 'initially non-Sold: repeat observation'
    controls = followup_plan.role.eq('historical_control') | followup_plan.first_encountered_sold
    followup_plan.loc[controls, 'followup_group'] = 'initially Sold or historical control'
    first_check = followup_plan.checks.eq(0)
    no_native = followup_plan.checks.gt(0) & followup_plan.last_usable_native_at.isna()
    followup_plan.loc[no_native, 'followup_group'] = 'attempted: no usable native evidence'
    followup_plan.loc[first_check, 'followup_group'] = 'no visit'
    reasons = {
        'no visit': 'Establish the first VIN-matched native website observation',
        'attempted: no usable native evidence': 'Review unsuccessful attempts before establishing a native baseline',
        'initially non-Sold: repeat observation': 'Observe a possible status change after a known native non-Sold boundary',
        'initially Sold or historical control': 'Check control stability or reappearance; not a new prospective sale',
    }
    followup_plan['check_reason'] = followup_plan.followup_group.map(reasons)
    # The physical time of the last usable native check sets the repeat clock.
    followup_plan['next_due_at'] = pd.to_datetime(followup_plan.last_usable_native_at,
        utc=True, format='ISO8601') + pd.Timedelta(hours=PILOT_RECHECK_HOURS)
    followup_plan['repeat_due'] = followup_plan.next_due_at.notna() & (
        followup_plan.next_due_at.le(pd.Timestamp(PILOT_AS_OF)))
    failed_latest = followup_plan.checks.gt(0) & followup_plan.latest_parse_outcome.ne('matched')
    has_url = followup_plan.followup_url.fillna('').str.strip().ne('')
    followup_plan['needs_review'] = no_native | failed_latest | ~has_url
    followup_plan['eligible_for_batch'] = (first_check | followup_plan.repeat_due) & ~followup_plan.needs_review
    followup_plan['deferred_reason'] = ''
    followup_plan.loc[~first_check & ~followup_plan.repeat_due, 'deferred_reason'] = 'Not yet due; wait until next_due_at'
    followup_plan.loc[failed_latest, 'deferred_reason'] = 'Review failed/conflicting latest attempt before retry; native clock unchanged'
    followup_plan.loc[no_native, 'deferred_reason'] = 'Review attempted checks with no usable native evidence; baseline still needed'
    followup_plan.loc[~has_url, 'deferred_reason'] = 'Review unresolved follow-up identity before checking'

    followup_plan['_batch_order'] = 2  # Deferred rows remain visible after eligible rows.
    followup_plan.loc[followup_plan.eligible_for_batch & first_check, '_batch_order'] = 0
    followup_plan.loc[followup_plan.eligible_for_batch & ~first_check, '_batch_order'] = 1
    followup_plan = followup_plan.sort_values(
        ['_batch_order', 'next_due_at', 'retailer', 'vin', 'followup_listing_id'],
        na_position='first', kind='stable').drop(columns='_batch_order')
    followup_plan['plan_rank'] = range(1, len(followup_plan) + 1)
    selected_index = followup_plan.loc[followup_plan.eligible_for_batch].head(FOLLOWUP_BATCH_LIMIT).index
    followup_plan['selected_for_batch'] = followup_plan.index.isin(selected_index)
    followup_plan.loc[followup_plan.eligible_for_batch & ~followup_plan.selected_for_batch,
        'deferred_reason'] = 'Eligible, but single-pass batch limit reached'
    next_followup_batch = followup_plan.loc[followup_plan.selected_for_batch].copy()
    followup_remainder = followup_plan.loc[~followup_plan.selected_for_batch].copy()
    followup_group_counts = followup_plan.groupby(['followup_group', 'selection_pending_group'],
        sort=True, dropna=False).size().rename('VINs').reset_index()
    followup_columns = ['plan_rank', 'cohort_id', 'retailer', 'vin', 'selection_pending_group',
        'followup_group', 'check_reason', 'last_usable_native_at', 'next_due_at', 'repeat_due',
        'needs_review', 'eligible_for_batch', 'deferred_reason', 'followup_listing_id', 'followup_url']
    print('Coverage groups at cutoff:')
    display(followup_group_counts)
    print('Selected:', len(next_followup_batch), '| Eligible:', int(followup_plan.eligible_for_batch.sum()),
          '| Single-pass limit:', FOLLOWUP_BATCH_LIMIT, '| Full cohort:', len(followup_plan))
    print('First checks, then oldest due repeats. Stop on challenge, rate limit or identity conflict.')
    with pd.option_context('display.max_rows', None, 'display.max_colwidth', 90):
        display(next_followup_batch[followup_columns])
        print('Remainder: not yet due, review needed, or batch limit reached:')
        display(followup_remainder[followup_columns])


The next manual step is to inspect this attention list and choose the next
checks from the fixed cohorts. A prior access challenge must be resolved before
resuming visits; a reminder is not permission to retry it. Keep failed and
unvisited cases visible.

Follow the [capture/import guide](../docs/sale_pilot.md) to capture a public page,
preview its VIN/listing identity and interpretation, and deliberately save the
observation. The importer preserves `checked_at` and records its actual import
availability. It does not write inventory, canonical checks or analyst reviews.
Run this notebook again to read the new evidence at a suitable cutoff.

No new visit is needed to use the tables above. A new website observation still
does not by itself establish delivery, completed sales, or forecast accuracy.


## 5. One vehicle's evidence timeline

Choose retailer/VIN in Settings. Read each physical check time beside its evidence
availability, listing ID, native fields, exact badge and retained source. The first
non-Sold-to-Sold interval is bounded by observations, not an inferred transaction
time. Historical report events are kept in the optional study below.


In [ ]:
vehicle_evidence_timeline = pd.DataFrame()
print('Selected retailer/VIN:', INSPECT_RETAILER, INSPECT_HISTORY_VIN)
if not pilot_observations.empty:
    vehicle_evidence_timeline = pilot_observations.loc[
        pilot_observations.retailer.eq(INSPECT_RETAILER) & pilot_observations.vin.eq(INSPECT_HISTORY_VIN)
    ].sort_values(['checked_at', 'available_at']).copy()
    vehicle_evidence_timeline['hours_since_previous_check'] = vehicle_evidence_timeline.checked_at.diff().dt.total_seconds() / 3600
    display(vehicle_evidence_timeline[['retailer', 'vin', 'listing_id', 'checked_at', 'available_at',
        'hours_since_previous_check', 'saleStatus', 'purchaseType', 'hero_badge', 'observed_status', 'parse_outcome', 'source']])
if vehicle_evidence_timeline.empty:
    print('No retained native check for this retailer/VIN at the cutoff; choose an identity from pilot_queue.')


## Optional A. Historical CARFAX/title study and inventory audit

These are **three different kinds of evidence**: inventory observations, website
wording, and events reported by a history provider. We join by retailer/VIN while
keeping each listing ID. A report on one listing cannot establish another listing's sale.

The bounded [source study](../docs/vehicle_history_findings.md) selected five frozen-cohort
cases and one **separate public Kia example**. It stopped at a CARFAX device challenge.
Report rows are selected public fields, not original HTTP bytes or a complete history.
No study evidence is imported into operating checks, reviews, or SQLite.

| Column | Meaning |
| --- | --- |
| `event_date` | Provider's event date, day precision; not our discovery date |
| `report_run_at` | Report generation time, converted to UTC |
| `first_observed_at` | First retrieval in this study, UTC; earlier availability unknown |
| `available_at` | When this project's retained evidence was available, UTC |
| `event_source` / `native_wording` | Provider's reporting source and exact event wording |
| `last_asking_price_usd` | Last observed website asking price in USD, never transaction price |

The following existing calculation retrieves inventory from read-only SQLite and
checks its coverage and identities. The result is available as `inventory_page_comparison`.

`website_status` / `page_listing_id` / `website_checked_at` describe the native pilot
visit. `study_website_status` / `study_listing_id` / `study_checked_at` describe a
separate manually retained history-study page observation. Neither is an analyst
confirmation; study observations are not promoted to native pilot captures.


In [ ]:
inventory_page_comparison = inventory_comparison_diagnostics = pd.DataFrame()
if not pilot_summary.empty:
    comparisons, diagnostics, validation = [], [], {}
    for page in pilot_summary.loc[pilot_summary.checks.gt(0)].itertuples():
        page_time = _aware(page.checked_at)
        item = dict(cohort_id=page.cohort_id, retailer=page.retailer, vin=page.vin, role=page.role,
            page_listing_id=page.latest_listing_id, page_checked_at=page_time,
            page_available_at=page.available_at, saleStatus=page.latest_saleStatus,
            purchaseType=page.latest_purchaseType, page_interpretation=page.latest_status,
            inventory_context='no_inventory_available_by_check', inventory_listing_id=None,
            inventory_observed_at=None, inventory_available_at=None, purchase_pending=None,
            asking_price_usd=None, elapsed_observation_hours=None, latest_inventory_cycle_date=None,
            latest_inventory_coverage_complete=None, inventory_source=None)
        if not inventory_cycles.empty:
            # Availability and physical clocks are separate restrictions, before any identity join.
            known_cycles = inventory_cycles.loc[inventory_cycles.available_at.map(_aware).le(page_time)]
            known_rows = inventory_source_rows.loc[inventory_source_rows.inventory_available_at.le(page_time)
                & inventory_source_rows.inventory_observed_at.le(page_time)]
            cycle_key = tuple(known_cycles.cycle_id)
            if cycle_key not in validation:
                try:
                    vin_events(known_cycles, known_rows)  # Reuse timeline's whole-population identity/window checks.
                    validation[cycle_key] = None
                except ValueError as error:
                    validation[cycle_key] = str(error)
            bindings = known_rows.loc[known_rows.retailer.eq(page.retailer) & known_rows.listing_id.eq(page.latest_listing_id)]
            problem = validation[cycle_key]
            if bindings.vin.ne(page.vin).any():
                problem = 'Page listing ID is bound to a different VIN in preceding inventory'
            if problem:
                item['inventory_context'] = 'invalid_identity_or_cycle_evidence'
                diagnostics.append(dict(cohort_id=page.cohort_id, vin=page.vin, problem=problem))
            elif not known_cycles.empty:
                latest_cycle = known_cycles.sort_values('cycle_date').iloc[-1]
                preceding = known_rows.loc[known_rows.retailer.eq(page.retailer) & known_rows.vin.eq(page.vin)]
                item.update(latest_inventory_cycle_date=latest_cycle.cycle_date,
                    latest_inventory_coverage_complete=latest_cycle.coverage_complete)
                if not preceding.empty:
                    previous = preceding.sort_values(['inventory_observed_at', 'inventory_available_at']).iloc[-1]
                    item.update(inventory_listing_id=previous.listing_id,
                        inventory_observed_at=previous.inventory_observed_at,
                        inventory_available_at=previous.inventory_available_at,
                        purchase_pending=previous.purchase_pending, asking_price_usd=previous.asking_price_usd,
                        elapsed_observation_hours=(page_time - previous.inventory_observed_at).total_seconds() / 3600,
                        inventory_source=previous.get('source_path', previous.get('source_url')))
                if preceding.empty and page.role == 'historical_control':
                    context = 'historical_control_not_observed_in_scope'
                elif latest_cycle.cycle_date != page_time.tz_convert(latest_cycle.timezone).date().isoformat():
                    context = 'collection_gap_unassessable'
                elif not latest_cycle.coverage_complete:
                    context = 'partial_cycle_unassessable'
                elif preceding.empty:
                    context = 'scope_membership_not_established'
                elif preceding.cycle_id.eq(latest_cycle.cycle_id).any():
                    context = 'observed_in_latest_complete_cycle'
                else:
                    context = 'not_observed_in_complete_cycle'
                item['inventory_context'] = context
        comparisons.append(item)
    inventory_page_comparison = pd.DataFrame(comparisons)
    inventory_comparison_diagnostics = pd.DataFrame(diagnostics)
    if SHOW_AUDIT_DETAILS:
        print('Preceding inventory and latest page attempt; both original clocks stay visible:')
        with pd.option_context('display.max_rows', None, 'display.max_columns', None):
            display(inventory_page_comparison)
    else:
        print('Preceding inventory audit is available in inventory_page_comparison.')
    if not inventory_comparison_diagnostics.empty:
        display(inventory_comparison_diagnostics)


In [ ]:
from vehicle_tracker.vehicle_history import load_report_events
from vehicle_tracker.sales import sale_candidates

report_events = load_report_events(HISTORY_STUDY_DIR, as_of=PILOT_AS_OF) if not alternative_view else pd.DataFrame()
study_pages = pd.DataFrame()
page_file = HISTORY_STUDY_DIR / 'detail_observations.json'
if not alternative_view and page_file.is_file():
    study_pages = pd.DataFrame(json.loads(page_file.read_text(encoding='utf-8')))
    for column in ['checked_at', 'available_at']:
        study_pages[column] = study_pages[column].map(_aware)
    study_pages = study_pages.loc[study_pages.checked_at.le(_aware(PILOT_AS_OF))
        & study_pages.available_at.le(_aware(PILOT_AS_OF))].copy()

report_diagnostics = report_events.loc[~report_events.identity_match] if not report_events.empty else report_events
if not report_diagnostics.empty:
    print('Unreadable or mismatched report identity: no corroboration assigned.')
    display(report_diagnostics[['listing_id', 'expected_vin', 'report_vin', 'report_outcome', 'context', 'event_note']])
if report_events.empty:
    print('No report evidence available at this cutoff; absence of events says nothing about sales.')
else:
    print('Reports known at cutoff (one row per retrieval, not per sale):')
    display(report_events[['report_id', 'listing_id', 'report_vin', 'identity_match',
        'report_run_at', 'first_observed_at', 'available_at']].drop_duplicates())


**Absence sensitivity: 3 versus 7 consecutive complete collection days.**
Three days is the current candidate default; seven is an analytical comparison.
MarketCheck also uses cross-domain listing history and dealer attribution, which
we do not have. This is not a reproduction of its method, and neither threshold
confirms a sale. Missing dates and partial collections break continuity.

The loop reuses the existing candidate rule, keeping the two episode tables in
`absence_candidates_by_days`. The visible comparison requires a current complete
collection and a sufficient uninterrupted streak; otherwise it says **not yet evaluable**.
A historical candidate remains in its episode table even after a gap or reappearance.


In [ ]:
absence_candidates_by_days = {}
absence_sensitivity = pd.DataFrame()
latest_inventory_events = pd.DataFrame()
comparison_keys = ['retailer', 'vin']
sensitivity_parts = []
if not pilot_queue.empty:
    if not inventory_cycles.empty:
        try:
            inventory_events = vin_events(inventory_cycles, inventory_rows)
            latest_inventory_events = inventory_events.sort_values('cycle_date').drop_duplicates(comparison_keys, keep='last')
        except ValueError as error:
            print('Inventory sensitivity unresolved:', error)
    for threshold in ABSENCE_SENSITIVITY_DAYS:
        result = pilot_queue[comparison_keys].copy()
        result['absence_days'] = threshold
        result['assessment'] = 'not yet evaluable'
        result['reason'] = 'No valid scoped inventory history for this VIN'
        result['consecutive_complete_absent_days'] = pd.Series(pd.NA, index=result.index, dtype='Int64')
        if not latest_inventory_events.empty:
            candidates, calendar = sale_candidates(inventory_cycles, inventory_rows,
                as_of=PILOT_AS_OF, absence_days=threshold)
            absence_candidates_by_days[threshold] = candidates
            columns = [*comparison_keys, 'observed_in_cycle', 'absence_streak', 'coverage_complete',
                       'cycle_date', 'timezone', 'timing_uncertain']
            result = result.merge(latest_inventory_events[columns], on=comparison_keys, how='left', validate='one_to_one')
            result['consecutive_complete_absent_days'] = result.absence_streak.astype('Int64')
            current_dates = result.timezone.map(lambda zone: _aware(PILOT_AS_OF).tz_convert(zone).date().isoformat()
                if pd.notna(zone) else None)
            current = result.cycle_date.eq(current_dates) & result.coverage_complete.eq(True)
            present = current & result.observed_in_cycle.eq(True)
            absent = current & result.observed_in_cycle.eq(False)
            qualifies = absent & result.absence_streak.ge(threshold) & result.timing_uncertain.eq(False)
            result.loc[result.cycle_date.notna(), 'reason'] = 'Missing or incomplete latest collection; continuity unknown'
            result.loc[absent, 'reason'] = 'Too few consecutive complete absent days or earlier coverage gap'
            result.loc[present, ['assessment', 'reason']] = ['observed in latest complete cycle', 'No current absence candidate']
            result.loc[qualifies, ['assessment', 'reason']] = ['persistent-absence candidate', 'Uninterrupted complete-day threshold met; not a confirmed sale']
        sensitivity_parts.append(result)
    absence_sensitivity = pd.concat(sensitivity_parts, ignore_index=True)
    print('Sensitivity for the selected cohort (not an estimate of sales):')
    display(absence_sensitivity.groupby(['absence_days', 'assessment', 'reason'], dropna=False).size().rename('VINs').reset_index())


In [ ]:
vehicle_history_comparison = public_history_example = pd.DataFrame()
if not pilot_summary.empty:
    columns = ['cohort_id', 'retailer', 'vin', 'listing_id', 'latest_listing_id', 'latest_status',
        'checked_at', 'first_encountered_sold', 'newly_observed_sold', 'last_non_sold_at', 'first_sold_at']
    vehicle_history_comparison = pilot_summary[columns].rename(columns={
        'listing_id': 'original_listing_id', 'latest_listing_id': 'page_listing_id',
        'latest_status': 'website_status', 'checked_at': 'website_checked_at'})
    for threshold in ABSENCE_SENSITIVITY_DAYS:
        sensitivity = absence_sensitivity.loc[absence_sensitivity.absence_days.eq(threshold),
            [*comparison_keys, 'assessment']].rename(columns={'assessment': f'absence_{threshold}_days'})
        vehicle_history_comparison = vehicle_history_comparison.merge(sensitivity, on=comparison_keys,
            how='left', validate='one_to_one')
    if not inventory_source_rows.empty:
        last_inventory = inventory_source_rows.sort_values(['inventory_observed_at', 'inventory_available_at']).drop_duplicates(comparison_keys, keep='last')
        last_inventory = last_inventory[[*comparison_keys, 'listing_id', 'asking_price_usd',
            'inventory_observed_at', 'capture_id', 'source_url']].rename(columns={
            'listing_id': 'last_inventory_listing_id', 'asking_price_usd': 'last_asking_price_usd',
            'source_url': 'inventory_source'})
        vehicle_history_comparison = vehicle_history_comparison.merge(last_inventory, on=comparison_keys,
            how='left', validate='one_to_one')
    if not study_pages.empty:
        cohort_pages = study_pages.loc[study_pages.cohort_member,
            [*comparison_keys, 'listing_id', 'website_status', 'checked_at', 'source_url']].rename(columns={
            'listing_id':'study_listing_id', 'website_status':'study_website_status',
            'checked_at':'study_checked_at', 'source_url':'study_page_source'})
        vehicle_history_comparison = vehicle_history_comparison.merge(cohort_pages, on=comparison_keys,
            how='left', validate='one_to_one')
    if not report_events.empty:
        # Keep wrong VINs in report_diagnostics, never attach their events to the expected vehicle.
        matching_events = report_events.loc[report_events.identity_match].rename(columns={
            'expected_vin':'vin', 'listing_id':'report_listing_id'})
        vehicle_history_comparison = vehicle_history_comparison.merge(matching_events, on=comparison_keys,
            how='left', validate='one_to_many')
    for column in ['event_date', 'event_kind', 'native_wording', 'report_listing_id',
        'event_source', 'source_url', 'inventory_observed_at', 'last_asking_price_usd',
        'study_listing_id', 'study_website_status', 'study_checked_at']:
        if column not in vehicle_history_comparison:
            vehicle_history_comparison[column] = pd.NA
    vehicle_history_comparison['interpretation'] = 'No usable report event at cutoff; sale remains unresolved'
    has_event = vehicle_history_comparison.event_date.notna()
    vehicle_history_comparison.loc[has_event, 'interpretation'] = 'Reported event; seller and listing-period attribution unresolved'
    for kind, meaning in {
        'title': 'Title record; may reflect financing or administration, not proof of retail sale',
        'registration': 'Registration or renewal; not proof of a new owner or retail sale',
        'auction_sale': 'Auction sale reported; not automatically a Carvana retail sale',
        'auction_listing': 'Auction appearance reported; not proof of a completed sale',
        'dealer_listing': 'Dealer inventory reported; dealer not identified as Carvana',
        'owner_change': 'Ownership change reported; listing-period and seller attribution still needed',
    }.items():
        vehicle_history_comparison.loc[vehicle_history_comparison.event_kind.eq(kind), 'interpretation'] = meaning
    # Compare dates at their documented precision. Same-day ordering is unknown.
    inventory_dates = pd.to_datetime(vehicle_history_comparison.inventory_observed_at, utc=True).dt.strftime('%Y-%m-%d')
    before_inventory = has_event & vehicle_history_comparison.event_date.lt(inventory_dates)
    vehicle_history_comparison.loc[before_inventory, 'interpretation'] += '; predates our last inventory observation'
    print('One row per report event; repeated VINs below are not repeated sales.')
    display_columns = ['vin', 'page_listing_id', 'website_status', 'website_checked_at',
        'study_listing_id', 'study_website_status', 'study_checked_at',
        'absence_3_days', 'absence_7_days', 'report_listing_id', 'event_date', 'event_source',
        'native_wording', 'interpretation']
    display(vehicle_history_comparison[display_columns])

if not report_events.empty and not study_pages.empty:
    demonstration = study_pages.loc[~study_pages.cohort_member]
    public_history_example = demonstration.merge(report_events.loc[report_events.identity_match],
        left_on=['retailer', 'vin', 'listing_id'], right_on=['retailer', 'report_vin', 'listing_id'],
        how='left', validate='one_to_many', suffixes=('_page', '_report'))
    print('Separate public Kia example: no scoped inventory denominator or sale transition interval.')
    display(public_history_example[['vin', 'listing_id', 'website_status', 'event_date', 'event_source', 'native_wording_report', 'context']])


**Inspect one vehicle.** Change the VIN in Settings, then follow its inventory capture,
website observations and report source. Compare the reported event date with our
actual inventory and page-observation times before interpreting it.

A report can describe multiple ownership or auction episodes. A later auction
cannot automatically be attributed to an earlier Carvana listing. Read `context`
for month-precision ownership summaries; do not invent an exact ownership-change day.
Missing events never establish that a vehicle remained unsold. Analyst-confirmed
outcomes still require explicit review; this notebook writes none.


In [ ]:
print('Selected VIN:', INSPECT_HISTORY_VIN)
if not inventory_source_rows.empty:
    display(inventory_source_rows.loc[inventory_source_rows.vin.eq(INSPECT_HISTORY_VIN),
        ['vin', 'listing_id', 'inventory_observed_at', 'inventory_available_at',
         'asking_price_usd', 'capture_id', 'source_url']])
if not pilot_observations.empty:
    display(pilot_observations.loc[pilot_observations.vin.eq(INSPECT_HISTORY_VIN),
        ['vin', 'listing_id', 'checked_at', 'available_at', 'saleStatus', 'purchaseType',
         'observed_status', 'source']])
if not report_events.empty:
    display(report_events.loc[report_events.expected_vin.eq(INSPECT_HISTORY_VIN),
        ['expected_vin', 'report_vin', 'listing_id', 'event_date', 'native_wording',
         'report_run_at', 'first_observed_at', 'available_at', 'source_url', 'source_file', 'event_note']])

# Reuse the existing read-only history reader to locate the retained source.
from vehicle_tracker.history import read_history, file_hash
selected_inventory = inventory_source_rows.loc[inventory_source_rows.vin.eq(INSPECT_HISTORY_VIN)] if not inventory_source_rows.empty else pd.DataFrame()
database = inventory_settings.get('database') if 'inventory_settings' in globals() else None
if not selected_inventory.empty and database is not None and Path(database).is_file():
    latest = selected_inventory.sort_values('inventory_observed_at').iloc[-1]
    _, capture_index, stored_rows = read_history(database, run_ids=[latest.run_id])
    source_rows = capture_index.loc[capture_index.capture_id.eq(latest.capture_id)]
    if len(source_rows) != 1:
        raise ValueError('Selected inventory capture must have one source reference.')
    retained_path = Path(source_rows.iloc[0].source_path)
    if file_hash(retained_path) != latest.capture_id:
        raise ValueError('Retained inventory capture hash changed.')
    retained = json.loads(retained_path.read_text(encoding='utf-8'))
    source_vehicle = [row for row in retained['vehicles']
        if row['vin'] == latest.vin and str(row['vehicleId']) == latest.listing_id]
    if len(source_vehicle) != 1:
        raise ValueError('Expected one matching listing/VIN in the retained projection.')
    print('Original retained inventory projection:', retained_path)
    display(pd.DataFrame(source_vehicle)[['vin', 'vehicleId', 'price', 'isPurchasePending']])
    print('SQLite observation corresponding to that capture:')
    display(stored_rows.loc[stored_rows.capture_id.eq(latest.capture_id) & stored_rows.vin.eq(latest.vin),
        ['vin', 'listing_id', 'asking_price_usd', 'purchase_pending', 'observed_at_utc', 'capture_id']])


## Optional B. Historical September 10 follow-up accounting

The recorded pass below is separate from the **next** batch above. Its saved plan
keeps the original selection order and the 21 cases not visited in that pass.
Coverage is recomputed from the same frozen cohorts at the pre-pass cutoff and
at this notebook's cutoff. That comparison can include later passes and is not
the increment attributable solely to the old batch; the exact pass rows are separate. New evidence cannot appear before its import availability.

`VINs with usable native evidence` includes native Available/Reservable even when
purchase readiness is unresolved. `Initially non-Sold prospective VINs with repeats`
requires a later usable native observation, not merely two attempted visits.
The transition and reappearance tables in section 3 keep their original meanings:
first native non-Sold-to-Sold intervals are website evidence, not delivery dates;
repeated Sold checks and initially Sold vehicles do not create extra sales.

The unresolved-status count excludes the resolved website word `unavailable`: its
underlying cause still needs review. Purchase readiness, reasons for unavailability,
and economic-sale confirmation are separate uncertainties.


In [ ]:
followup_pass_coverage = followup_pass_changes = pd.DataFrame()
pass_file = FOLLOWUP_PASS_DIR / 'pass.json'
if not alternative_view and pass_file.is_file():
    followup_pass = json.loads(pass_file.read_text(encoding='utf-8'))
    if _aware(followup_pass['available_at']) <= _aware(PILOT_AS_OF):
        saved_run = ROOT / followup_pass['saved_run']
        if file_hash(saved_run / 'run.json') != followup_pass['run_manifest_sha256']:
            raise ValueError('The recorded pilot-run manifest changed.')
        accounting_fields = ['started_at', 'finished_at', 'available_at', 'elapsed_seconds',
            'detail_visits', 'report_visits', 'navigation_retries', 'rate_limits_or_challenges_observed',
            'identity_conflicts', 'matched_captures', 'usable_native_captures',
            'resolved_status_captures', 'minimum_start_spacing_seconds', 'stop_reason']
        display(pd.DataFrame([followup_pass])[accounting_fields].T.rename(columns={0: 'recorded_pass'}))
        print('Browser subrequests / HTTP status:', followup_pass['browser_subrequests'], '/', followup_pass['http_status'])
        print('Preserved original batch and remainder:', FOLLOWUP_PASS_DIR / 'plan.json')
        print('Saved pilot run:', saved_run)
        before_parts = []
        for cohort in known_cohorts:
            before_records = load_pilot(ROOT, cohort, as_of=followup_pass['baseline_as_of'])
            before_parts.append(summarize_pilot(before_records, cohort,
                as_of=followup_pass['baseline_as_of'], recheck_hours=PILOT_RECHECK_HOURS))
        before_summary = pd.concat(before_parts, ignore_index=True) if before_parts else pd.DataFrame()
        coverage_rows = []
        for label, summary in [('Before this pass', before_summary), ('At notebook cutoff', pilot_summary)]:
            if summary.empty:
                continue
            coverage_rows.append({'view': label, 'Selected VINs': len(summary),
                'VINs visited': int(summary.checks.gt(0).sum()),
                'No visit': int(summary.checks.eq(0).sum()),
                'Attempted, no usable native': int((summary.checks.gt(0) & summary.usable_native_checks.eq(0)).sum()),
                'VINs with usable native evidence': int(summary.usable_native_checks.gt(0).sum()),
                'Initially non-Sold prospective VINs with repeats': int(summary.prospective_repeat_observed.sum()),
                'Physical visits': int(summary.checks.sum()),
                'Usable native observations': int(summary.usable_native_checks.sum()),
                'Latest unresolved or failed status': int((summary.checks.gt(0)
                    & (~summary.latest_status.isin(['available', 'pending', 'unavailable', 'sold_label'])
                       | summary.latest_parse_outcome.ne('matched'))).sum()),
                'Initially Sold VINs': int(summary.first_encountered_sold.sum()),
                'First qualifying Sold transitions': int(summary.newly_observed_sold.sum()),
                'VINs with later reappearance': int(summary.reappeared_at.notna().sum())})
        followup_pass_coverage = pd.DataFrame(coverage_rows).set_index('view').T
        display(followup_pass_coverage)
        # Exact imported sources select this pass, rather than every check on the same day.
        pass_checks = pilot_observations.loc[pilot_observations.source.map(
            lambda source: Path(source).parent.resolve() == saved_run.resolve())]
        before_status = before_summary[['retailer', 'vin', 'latest_status', 'last_usable_native_at']].rename(
            columns={'latest_status': 'before_status', 'last_usable_native_at': 'before_last_usable_at'})
        followup_pass_changes = pass_checks.merge(before_status, on=['retailer', 'vin'],
            how='left', validate='one_to_one')
        display(followup_pass_changes[['vin', 'listing_id', 'before_status', 'before_last_usable_at',
            'checked_at', 'available_at', 'saleStatus', 'purchaseType', 'observed_status', 'parse_outcome']])
    else:
        print('The follow-up pass was not yet available at this cutoff.')
else:
    print('No recorded follow-up pass available in this optional experimental folder.')


## Optional C. Full membership and retained-source audit

The inventory calculations above and source rows below help check a surprising row.
Set `SHOW_AUDIT_DETAILS_OVERRIDE = True` before Run All to display the full tables.

`inventory_page_comparison` joins the latest page attempt to preceding inventory
by retailer/VIN. Both the inventory observation and its cycle availability must
precede the page check. Page and inventory listing IDs remain separate. A missing
date, partial cycle, or historical control outside the inventory scope cannot
establish absence or a sale; inspect `inventory_context` and the elapsed hours.

`pilot_queue` preserves fixed membership and selection reasons. `sale_signal_study`
contains all retained versions; `pilot_observations` uses the latest available
interpretation per physical visit. Failed checks keep their missing native fields.


In [ ]:
if SHOW_AUDIT_DETAILS:
    for cohort in known_cohorts:
        print('FROZEN MEMBERSHIP:', cohort['cohort_id'])
        with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
            display(pilot_queue.loc[pilot_queue.cohort_id.eq(cohort['cohort_id'])])
        if 'selection' in cohort:
            print('Frozen sampling metadata, not a new selection:')
            selection = cohort['selection']
            selection_age_hours = (pd.Timestamp(cohort['selected_at']) - pd.Timestamp(selection['source']['window_end'])).total_seconds() / 3600
            display(pd.DataFrame([dict(seed=selection['seed'], method=selection['method'],
                pandas_version=selection['pandas_version'], source_age_at_selection_hours=selection_age_hours)]))
            display(pd.DataFrame(selection['counts']))
            display(pd.DataFrame([cohort['selection']['source']]))
    if not sale_signal_study.empty:
        print('All retained versions at the cutoff; physical visits are deduplicated for coverage:')
        with pd.option_context('display.max_rows', None, 'display.max_columns', None):
            display(sale_signal_study[['cohort_id', 'retailer', 'vin', 'listing_id', 'observed_listing_id',
                'observed_vin', 'checked_at', 'available_at', 'saleStatus', 'purchaseType', 'inventoryType',
                'hero_badge', 'purchase_button', 'observed_status', 'access_outcome', 'parse_outcome', 'source']])
    else:
        print('No retained observations for the known cohorts at this cutoff.')
else:
    print('Membership and full source rows remain available in pilot_queue and sale_signal_study.')
